<a href="https://colab.research.google.com/github/aasimk910/AI-Module/blob/main/Worksheetweek6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/AI-ML/Data/week6/FruitinAmazon (1).zip"
extract_path = "/content/FruitinAmazon"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
os.listdir("/content/FruitinAmazon/FruitinAmazon/test")

['pupunha', 'cupuacu', 'acai', 'graviola', 'tucuma', 'guarana']

In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    "/content/FruitinAmazon/FruitinAmazon/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    "/content/FruitinAmazon/FruitinAmazon/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    "/content/FruitinAmazon/FruitinAmazon/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 72 images belonging to 6 classes.
Found 18 images belonging to 6 classes.
Found 30 images belonging to 6 classes.


In [8]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 13s 4s/step - accuracy: 0.1944 - loss: 2.5154 - val_accuracy: 0.3333 - val_loss: 1.6023
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 930ms/step - accuracy: 0.3472 - loss: 1.7198 - val_accuracy: 0.4444 - val_loss: 1.1852
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 978ms/step - accuracy: 0.6389 - loss: 0.9781 - val_accuracy: 0.6111 - val_loss: 0.8922
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.7222 - loss: 0.7900 - val_accuracy: 0.7778 - val_loss: 0.6608
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.8611 - loss: 0.5215 - val_accuracy: 0.8889 - val_loss: 0.5234
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.8333 - loss: 0.3999 - val_accuracy: 0.9444 - val_loss: 0.4207
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.9444 - loss: 0.2746 - val_accuracy: 0.9444 - val_loss: 0.3641
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.9722 - loss: 0.2104 - val_accuracy: 0.9444 - val_loss: 0.3518
Epoch 9/1

In [10]:
from sklearn.metrics import classification_report
import numpy as np

predictions = model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
              precision    recall  f1-score   support

        acai       1.00      0.80      0.89         5
     cupuacu       1.00      1.00      1.00         5
    graviola       1.00      1.00      1.00         5
     guarana       0.71      1.00      0.83         5
     pupunha       0.83      1.00      0.91         5
      tucuma       1.00      0.60      0.75         5

    accuracy                           0.90        30
   macro avg       0.92      0.90      0.90        30
weighted avg       0.92      0.90      0.90        30



In [11]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/FruitinAmazon/FruitinAmazon/train/pupunha/images (1).jpeg"  # change this

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Prediction: pupunha


In [12]:
from tensorflow.keras import Sequential

scratch_model = Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

scratch_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

scratch_model.fit(train_data, validation_data=val_data, epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.1111 - loss: 19.5371 - val_accuracy: 0.2222 - val_loss: 4.3747
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.3333 - loss: 2.8435 - val_accuracy: 0.2222 - val_loss: 2.4329
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 2s/step - accuracy: 0.2639 - loss: 2.1073 - val_accuracy: 0.3889 - val_loss: 1.6977
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 18s 5s/step - accuracy: 0.5139 - loss: 1.5131 - val_accuracy: 0.3889 - val_loss: 1.5953
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.6944 - loss: 1.1795 - val_accuracy: 0.5000 - val_loss: 1.4554
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.6806 - loss: 0.8948 - val_accuracy: 0.4444 - val_loss: 1.4372
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 4s/step - accuracy: 0.7500 - loss: 0.6369 - val_accuracy: 0.5000 - val_loss: 1.1360
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 3s/step - accuracy: 0.8472 - loss: 0.4652 - val_accuracy: 0.5000 - val_loss: 0.9662
Epoch 9/10
3/3

In [13]:
predictions = scratch_model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 698ms/step
              precision    recall  f1-score   support

        acai       1.00      0.40      0.57         5
     cupuacu       0.56      1.00      0.71         5
    graviola       1.00      0.80      0.89         5
     guarana       0.50      0.40      0.44         5
     pupunha       0.50      0.80      0.62         5
      tucuma       0.67      0.40      0.50         5

    accuracy                           0.63        30
   macro avg       0.70      0.63      0.62        30
weighted avg       0.70      0.63      0.62        30



In [14]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/FruitinAmazon/FruitinAmazon/train/pupunha/images (1).jpeg"  # change this

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = scratch_model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
Prediction: pupunha
